# Plant Health Checker (Healthy vs Diseased tea leaf)

Run each cell **one by one, from top to bottom**, by clicking the play button on the left of the cell.
Wait until a cell finishes (a green tick appears) before running the next one.

## Step 1: Upload your zip file

In [ ]:
from google.colab import files
import zipfile, os

print("Click 'Choose Files' and select your archive zip file (the tea leaf dataset)")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_name) as z:
    z.extractall(".")

print("Folders now available:", [f for f in os.listdir(".") if os.path.isdir(f)])

## Step 2: Make two groups: healthy and diseased

In [ ]:
import os, random, shutil

SOURCE_DIR = "Tea_Leaf_Disease"
OUTPUT_DIR = "data"
IMAGE_EXTS = (".jpg", ".jpeg", ".png")
random.seed(42)

def list_images(folder):
    return [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(IMAGE_EXTS)]

healthy_files = []
disease_folders = {}

for name in sorted(os.listdir(SOURCE_DIR)):
    path = os.path.join(SOURCE_DIR, name)
    if not os.path.isdir(path):
        continue
    files_in = list_images(path)
    if "healthy" in name.lower():
        healthy_files.extend(files_in)
    else:
        disease_folders[name] = files_in

target = len(healthy_files)
per_disease = target // len(disease_folders)

diseased_files = []
for name, files_in in disease_folders.items():
    random.shuffle(files_in)
    diseased_files.extend(files_in[:per_disease])

chosen = set(diseased_files)
leftovers = [f for fl in disease_folders.values() for f in fl if f not in chosen]
random.shuffle(leftovers)
diseased_files.extend(leftovers[: target - len(diseased_files)])

for label, file_list in [("healthy", healthy_files), ("diseased", diseased_files)]:
    out = os.path.join(OUTPUT_DIR, label)
    os.makedirs(out, exist_ok=True)
    for i, src in enumerate(file_list):
        shutil.copy(src, os.path.join(out, f"{label}_{i}.jpg"))
    print(f"{label}: {len(file_list)} images")

## Step 3: Train the model (takes a few minutes)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 8

train_ds = tf.keras.utils.image_dataset_from_directory(
    "data", validation_split=0.2, subset="training", seed=42,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="binary")

val_ds = tf.keras.utils.image_dataset_from_directory(
    "data", validation_split=0.2, subset="validation", seed=42,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="binary")

print("Classes:", train_ds.class_names)   # diseased = 0, healthy = 1

train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

base = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet")
base.trainable = False

inputs = layers.Input(shape=IMG_SIZE + (3,))
x = augment(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = models.Model(inputs, outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="binary_crossentropy", metrics=["accuracy"])

history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)
model.save("plant_model.keras")

plt.plot(history.history["accuracy"], label="train")
plt.plot(history.history["val_accuracy"], label="validation")
plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend()
plt.title("Plant Health Classifier")
plt.savefig("accuracy.png")
plt.show()

print("Final validation accuracy:", round(history.history["val_accuracy"][-1] * 100, 1), "%")

## Step 4: Test it with an image

Run the cell, click **Choose Files**, and pick a tea leaf photo.
You can run this cell again and again with different images.

In [ ]:
from google.colab import files
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

model = tf.keras.models.load_model("plant_model.keras")

up = files.upload()
for name in up.keys():
    img = Image.open(name).convert("RGB")
    arr = np.array(img.resize((224, 224)), dtype=np.float32)[None]
    p_healthy = float(model.predict(arr, verbose=0)[0][0])

    if p_healthy >= 0.5:
        result = f"HEALTHY ({p_healthy * 100:.1f}% sure)"
    else:
        result = f"NOT HEALTHY ({(1 - p_healthy) * 100:.1f}% sure)"

    plt.imshow(img)
    plt.axis("off")
    plt.title(result)
    plt.show()
    print(name, "->", result)